# Polymarket CLOB API

Uses the Gamma API once for market discovery (conditionId + token IDs), then hits the CLOB API for all live price data.

In [10]:
import requests
import time
import json

## 1. Discovery (Gamma API)

Used once to find the current open window and extract the CLOB identifiers.

In [11]:
# Slug encodes the window start as a Unix timestamp (5-min boundary)
window_start = (int(time.time()) // 300) * 300
slug = f"btc-updown-5m-{window_start}"

resp = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"slug": slug}
)
event = resp.json()[0]
market = event["markets"][0]

condition_id = market["conditionId"]
token_ids = json.loads(market["clobTokenIds"])
up_token, down_token = token_ids

print(f"Slug   : {slug}")
print(f"Market : {event['title']}")
print(f"End    : {market['endDate']}")
print(f"Condition ID : {condition_id}")
print(f"Up token     : {up_token}")
print(f"Down token   : {down_token}")

Slug   : btc-updown-5m-1778611200
Market : Bitcoin Up or Down - May 12, 2:40PM-2:45PM ET
End    : 2026-05-12T18:45:00Z
Condition ID : 0xc7d5038e561092af41b2aeb037ffea79847bcedaed7fad1231b81ce7cf87468f
Up token     : 31233271477707144524363299809869681313677129453987526296501664711489288326245
Down token   : 19443615647285359569003339903107144459193131336683259334537930888806188238577


In [12]:
resp.url

'https://gamma-api.polymarket.com/events?slug=btc-updown-5m-1778611200'

## 2. CLOB Market Info

In [13]:
r = requests.get(f"https://clob.polymarket.com/markets/{condition_id}")
clob_market = r.json()
clob_market.keys()

dict_keys(['enable_order_book', 'active', 'closed', 'archived', 'accepting_orders', 'accepting_order_timestamp', 'minimum_order_size', 'minimum_tick_size', 'condition_id', 'question_id', 'question', 'description', 'market_slug', 'end_date_iso', 'game_start_time', 'seconds_delay', 'fpmm', 'maker_base_fee', 'taker_base_fee', 'notifications_enabled', 'neg_risk', 'neg_risk_market_id', 'neg_risk_request_id', 'icon', 'image', 'rewards', 'is_50_50_outcome', 'tokens', 'tags'])

In [14]:
r.url

'https://clob.polymarket.com/markets/0xc7d5038e561092af41b2aeb037ffea79847bcedaed7fad1231b81ce7cf87468f'

## 3. Live Prices

In [15]:
# Midpoint price for "Up" outcome
r = requests.get("https://clob.polymarket.com/midpoint", params={"token_id": up_token})
print("Midpoint P(Up):", r.json())

Midpoint P(Up): {'mid': '0.575'}


In [16]:
# Best bid and ask
bid = requests.get("https://clob.polymarket.com/price", params={"token_id": up_token, "side": "BUY"}).json()
ask = requests.get("https://clob.polymarket.com/price", params={"token_id": up_token, "side": "SELL"}).json()
print("Bid:", bid)
print("Ask:", ask)

Bid: {'price': '0.56'}
Ask: {'price': '0.57'}


In [17]:
# Full order book (top of book)
r = requests.get("https://clob.polymarket.com/book", params={"token_id": up_token})
book = r.json()
print("Best bid:", book["bids"][0] if book.get("bids") else None)
print("Best ask:", book["asks"][0] if book.get("asks") else None)

Best bid: {'price': '0.01', 'size': '15596.66'}
Best ask: {'price': '0.99', 'size': '15434.6'}


## 4. Live Polling Loop

In [ ]:
for i in range(50):
    r = requests.get("https://clob.polymarket.com/midpoint", params={"token_id": up_token})
    mid = r.json().get("mid")
    print(f"[{time.asctime()}] P(Up) = {mid}")
    time.sleep(15)

[Tue May 12 20:40:28 2026] P(Up) = 0.575
[Tue May 12 20:40:43 2026] P(Up) = 0.625
[Tue May 12 20:40:59 2026] P(Up) = 0.645
[Tue May 12 20:41:14 2026] P(Up) = 0.595
[Tue May 12 20:41:30 2026] P(Up) = 0.575
[Tue May 12 20:41:45 2026] P(Up) = 0.555
[Tue May 12 20:42:00 2026] P(Up) = 0.705
[Tue May 12 20:42:16 2026] P(Up) = 0.805
[Tue May 12 20:42:31 2026] P(Up) = 0.855
[Tue May 12 20:42:46 2026] P(Up) = 0.855
[Tue May 12 20:43:02 2026] P(Up) = 0.845
[Tue May 12 20:43:17 2026] P(Up) = 0.935
[Tue May 12 20:43:32 2026] P(Up) = 0.955
[Tue May 12 20:43:48 2026] P(Up) = 0.975
[Tue May 12 20:44:03 2026] P(Up) = 0.995
[Tue May 12 20:44:18 2026] P(Up) = 0.995
[Tue May 12 20:44:33 2026] P(Up) = 0.995
[Tue May 12 20:44:49 2026] P(Up) = 0.995
[Tue May 12 20:45:04 2026] P(Up) = 0.995
[Tue May 12 20:45:19 2026] P(Up) = 0.995
[Tue May 12 20:45:35 2026] P(Up) = 0.995
[Tue May 12 20:45:50 2026] P(Up) = 0.995
[Tue May 12 20:46:05 2026] P(Up) = 0.995
[Tue May 12 20:46:20 2026] P(Up) = 0.995
[Tue May 12 20:4